In [1]:
import sys
import json
from tqdm import tqdm
from openai import OpenAI 
import os
from gigachat import GigaChat 
import joblib
from gigachat.models import Chat, Messages 
from functools import reduce
from typing import Dict
import gc

# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
# TO CHNAGE

sys.path.insert(0, BASEDIR)

from src.memorize_pipeline import MemPipeline
from src.memorize_pipeline.extractor.LLMExtractor import LLMExtractor
from src.memorize_pipeline.updator.LLMUpdator import LLMUpdator
from src.llm_agent import AgentConnector
from src.utils.data_structs import TripletCreator, NodeCreator, Relation, NODES_TYPES_MAP, RELATIONS_TYPES_MAP
from src.llm_agent.agent_model import SYSTEM_PROMPT

from src.neo4j_functions import Neo4jConnection
from src.embedding_functions import EmbeddingsDatabaseConnection, EmbeddingsDatabaseConnectionConfig, VectorDBConnectionConfig
from src.knowledge_graph_model import KnowledgeGraphModel

DATASET_PATH = '../data/Augment_DiaASQ.json'
SAVE_EXTRACTED_TRIPLETS_FILE = "tmp_extracted_gigachat_triplets.json"
gc.collect()

0

In [ ]:
class OpenAIAgent:

    def __init__(self, api_key, model: str = 'gpt-4o-mini') -> None:
        self.model = model
        self.client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", api_key))
        self.system_prompt = SYSTEM_PROMPT

    def generate(self, user_prompt: str, assistant_prompt: str = None, 
                 system_prompt: str = None, gen_strategy: Dict = None) -> str:
        messages = [
            {"role": "system", "content": system_prompt if system_prompt is not None else self.system_prompt},
            {"role": "user","content": user_prompt}]

        if assistant_prompt is not None:
            messages.insert(1, {"role": "assistant", "content": assistant_prompt})

        completion = self.client.chat.completions.create(
            model=self.model, messages=messages)

        return completion.choices[0].message.content

In [5]:
class GigaChatAgent:
    def __init__(self, creds: str, scope: str = 'GIGACHAT_API_CORP', model: str = "GigaChat-Pro",
                 verify_ssl_certs: bool = False) -> None:
        self.giga_model = GigaChat(
            credentials=creds, scope=scope, verify_ssl_certs=verify_ssl_certs, model=model) 
        self.system_prompt = SYSTEM_PROMPT

    def generate(self, user_prompt: str, assistant_prompt: str = None, 
                 system_prompt: str = None, gen_strategy: Dict = None) -> str:

        chat = Chat(messages=[Messages(role='system', content=system_prompt if system_prompt is not None else self.system_prompt), 
                            Messages(role='user', content=user_prompt)]) 
        response = self.giga_model.chat(chat) 
        return response.choices[0].message.content

In [6]:
GIGACHAT_CREDS = 'OWUwOGUzOWEtMjJiNi00YmMxLThmMmItNzMwNjM2MTI2YmYxOjg2ODdiOTVhLTZkNDctNGFjOC1iMmViLTEyNDA5MmFiN2Q5Mw=='
agent = GigaChatAgent(creds=GIGACHAT_CREDS)

In [ ]:
API_KEY = "'sk-861mINAavom2SSBqgrI82D4thMOfqT37knCof2o0H0T3BlbkFJ2gdVXJuVjNesNNP2aeUwPoBpZP3a3R1gn1kqv97CsA'"
agent = OpenAIAgent(api_key=API_KEY)

In [7]:
agent.generate("Сколько будет 2+2?")

'2 плюс 2 равно 4.'

In [8]:
with open(DATASET_PATH, 'r', encoding='utf-8') as fd:
    data = json.loads(fd.read())

In [9]:
raw_texts = list(map(lambda v: v['text_dialog'], data['data']))
raw_time = list(map(lambda v: v['time'].split(',')[0], data['data']))
print(len(raw_texts), len(raw_time))

3483 3483


In [10]:
extractor = LLMExtractor(agent_conn=agent)

In [19]:
extracted_triplets = []

In [20]:
for i in tqdm(range(len(raw_texts))):
    out = extractor.extract(raw_texts[i])
    extracted_triplets.append(out)

100%|██████████| 1/1 [00:20<00:00, 20.45s/it]


In [ ]:
# adding time
for group_idx in tqdm(range(len(extracted_triplets))):
    cur_time = raw_time[group_idx]
    for triplet_idx in range(len(extracted_triplets[group_idx])):
        if extracted_triplets[group_idx][triplet_idx].relation.prop['type'] == 'simple':
            extracted_triplets[group_idx][triplet_idx].relation.prop['time'] = cur_time
        else:
            extracted_triplets[group_idx][triplet_idx].end_node.prop['time'] = cur_time

In [ ]:
print(sum(list(map(len, extracted_triplets))))
joblib.dump(extracted_triplets, SAVE_EXTRACTED_TRIPLETS_FILE)